In [0]:
from pyspark.sql import functions as F

claims = spark.table("silver_claims")
members = spark.table("silver_member")

# Claims summary by region
gold_region_summary = (
    claims.join(members, "member_id")
    .groupBy(members["region"])
    .agg(
        F.count("claim_id").alias("total_claims"),
        F.round(F.sum("claim_amount"), 2).alias("total_claim_amount")
    )
)

gold_region_summary.write.mode("overwrite").format("delta").saveAsTable("gold_region_summary")

# Claims summary by status
gold_status_summary = (
    claims.groupBy("claim_status")
    .agg(
        F.count("claim_id").alias("total_claims"),
        F.round(F.sum("claim_amount"), 2).alias("total_claim_amount")
    )
)

gold_status_summary.write.mode("overwrite").format("delta").saveAsTable("gold_status_summary")

print("Gold tables created successfully!")

In [0]:
print("Silver claims:", spark.table("silver_claims").count())

print("Gold region summary:")
display(spark.table("gold_region_summary"))

print("Gold status summary:")
display(spark.table("gold_status_summary"))

print(
    "Total claims in region summary:",
    spark.table("gold_region_summary")
    .agg(F.sum("total_claims"))
    .first()[0]
)

print(
    "Total claims in status summary:",
    spark.table("gold_status_summary")
    .agg(F.sum("total_claims"))
    .first()[0]
)